In [ ]:
!pip install --upgrade lightgbm -q
!pip install --upgrade category-encoders
!pip install joblib

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from google.colab import drive

from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import lightgbm as lgb
import xgboost as xgb


from category_encoders import TargetEncoder

from sklearn.linear_model import LogisticRegression

import joblib

drive.mount('/content/drive', force_remount=True)
set_config(transform_output="pandas")

In [ ]:
PROJECT_PATH = '/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project'

X_train_full = pd.read_parquet(f'{PROJECT_PATH}/X_train_full.parquet')
y_train = pd.read_parquet(f'{PROJECT_PATH}/y_train.parquet')['TARGET']
X_test_full = pd.read_parquet(f'{PROJECT_PATH}/X_test_full.parquet')
test_id = pd.read_parquet(f'{PROJECT_PATH}/test_id.parquet')['SK_ID_CURR']

In [ ]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=42)

In [ ]:
pcb_cols = [ col for col in X_train_full if col.startswith("PREV_")]
bereua_cols = [ col for col in X_train_full if col.startswith("BUREAU_") and not col.startswith("BUREAU_BB_")]
custom_fill_na_cols = pcb_cols + bereua_cols

In [ ]:
cat_cols = set(X_train_full.select_dtypes(include = ['string', 'object', 'category']).columns)
cat_cols_list = list(cat_cols)

In [ ]:
def fillna_col (df):
    return df.fillna(0)
num_zero_transform = FunctionTransformer(fillna_col)
def to_category(df):
    return df.astype("category")
def fillna_category(df):
    return df.fillna("None")
cat_with_na = Pipeline([
    ("fill_na", FunctionTransformer(fillna_category)),
    ("to_category", FunctionTransformer(to_category))
])
def car_age(df):
    return df.fillna(-1)
fill_car_age = FunctionTransformer(car_age)
def fix_day_employed(df):
    df = df.copy()
    if "DAYS_EMPLOYED" in df.columns:
        df["DE_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
        df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace({365243: np.nan})
    return df
fix_de = FunctionTransformer(fix_day_employed)

def custom_feature_full(df):
    df = df.copy()

    monthly_income = df["AMT_INCOME_TOTAL"] / 12
    df["MONTHLY_INCOME_TO_PREV_CREDIT"] = monthly_income / (df["PREV_AMT_CREDIT_MEAN"] + 1)
    df["PREV_ANNUITY_TO_MONTHLY_INCOME"] = df["PREV_AMT_ANNUITY_MEAN"] / (monthly_income + 1)

    fix_member = df["CNT_FAM_MEMBERS"].fillna(1).replace(0, 1)
    df["INCOME_PER_FAMILY_MEMBER"] = df["AMT_INCOME_TOTAL"] / fix_member

    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

    e1 = df["EXT_SOURCE_1"].fillna(df["EXT_SOURCE_1"].median()).fillna(0.5)
    e2 = df["EXT_SOURCE_2"].fillna(df["EXT_SOURCE_2"].median()).fillna(0.5)
    e3 = df["EXT_SOURCE_3"].fillna(df["EXT_SOURCE_3"].median()).fillna(0.5)
    df["EXT_SOURCE_PROD"] = e1 * e2 * e3
    df["EXT_SOURCE_COUNT"] = df[ext_cols].notnull().sum(axis=1)

    df["PREV_DAYS_DECISION_SPAN"] = df["PREV_DAYS_DECISION_MAX"] - df["PREV_DAYS_DECISION_MIN"]

    df["BUREAU_DAYS_CREDIT_SPAN"] = df["BUREAU_DAYS_CREDIT_MAX"] - df["BUREAU_DAYS_CREDIT_MIN"]

    return df
custom_feature_step = FunctionTransformer(custom_feature_full)

In [ ]:
te_cols = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
target_encoder_step = TargetEncoder(
    smoothing = 10.0
)

In [ ]:
preprocessor = ColumnTransformer (
    transformers = [
        ("tar_encd_col", target_encoder_step, te_cols),
        ("cat_col", cat_with_na, cat_cols_list),
        ("custom_col", num_zero_transform, custom_fill_na_cols),
    ],
    remainder="passthrough",)

In [ ]:

model_lgbm = Pipeline([
    ("custom_feature", custom_feature_step),
    ("de_fix", fix_de),
    ("prep", preprocessor),
    ("LGBM", lgb.LGBMClassifier(
        random_state=17,
        n_jobs=4,
        verbose=-1
    ))
])

model_xgb = Pipeline([
    ("custom_feature", custom_feature_step),
    ("de_fix", fix_de),
    ("prep", preprocessor),
    ("XGBoost", xgb.XGBClassifier(
        tree_method="hist",
        device="cuda",
        random_state=17,
        verbosity=0,
        enable_categorical=True,
        eval_metric="auc"
    ))
])

In [ ]:
PARAMS_PATH = f'{PROJECT_PATH}/best_params.json'

with open(PARAMS_PATH, 'r') as f:
    best_params = json.load(f)

best_params_lgbm = best_params['params']['lgbm']
best_params_xgb = best_params['params']['xgb']

In [ ]:
oof_result_lgbm = np.zeros(len(X_train_full))
test_result_lgbm = np.zeros(len(X_test_full))
model_list_lgbm = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train), 1):
    X_tr, y_tr = X_train_full.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train_full.iloc[val_idx], y_train.iloc[val_idx]

    fold_model = clone(model_lgbm)
    fold_model.set_params(**{f"LGBM__{k}": v for k, v in best_params_lgbm.items()})
    fold_model.fit(X_tr, y_tr)

    val_preds = fold_model.predict_proba(X_val)[:, 1]
    oof_result_lgbm[val_idx] = val_preds

    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold} | ROC-AUC: {fold_auc:.5f}")

    test_preds = fold_model.predict_proba(X_test_full)[:, 1]
    test_result_lgbm += test_preds / skf.n_splits
    model_list_lgbm.append(fold_model)

overall_lgbm_auc = roc_auc_score(y_train, oof_result_lgbm)
print(f"Overall LGBM OOF ROC-AUC: {overall_lgbm_auc:.5f}")

In [ ]:
oof_result_xgb = np.zeros(len(X_train_full))
test_result_xgb = np.zeros(len(X_test_full))
model_xgb_list = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train), 1):
    X_tr, y_tr = X_train_full.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train_full.iloc[val_idx], y_train.iloc[val_idx]

    fold_model = clone(model_xgb)
    fold_model.set_params(**{f"XGBoost__{k}": v for k, v in best_params_xgb.items()})
    fold_model.fit(X_tr, y_tr)

    val_preds = fold_model.predict_proba(X_val)[:, 1]
    oof_result_xgb[val_idx] = val_preds

    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold} | ROC-AUC: {fold_auc:.5f}")

    test_preds = fold_model.predict_proba(X_test_full)[:, 1]
    test_result_xgb += test_preds / skf.n_splits
    model_xgb_list.append(fold_model)

overall_xgb_auc = roc_auc_score(y_train, oof_result_xgb)
print(f"Overall XGBoost OOF ROC-AUC: {overall_xgb_auc:.5f}")

In [ ]:
calibrator_lgbm = IsotonicRegression(out_of_bounds = 'clip', y_min = 0.0, y_max = 1.0)
calibrator_xgb = IsotonicRegression(out_of_bounds = 'clip', y_min = 0.0, y_max = 1.0)
calibrator_lgbm.fit(oof_result_lgbm, y_train)
calibrator_xgb.fit(oof_result_xgb, y_train)
oof_result_lgbm_calib = calibrator_lgbm.predict(oof_result_lgbm)
oof_result_xgb_calib = calibrator_xgb.predict(oof_result_xgb)
oof_blend_calib = 0.5 * oof_result_xgb_calib + 0.5 * oof_result_lgbm_calib
oof_blend_res = roc_auc_score(y_train, oof_blend_calib)
print(f"roc_auc of overall result: {oof_blend_res:.5f}")

In [ ]:
test_result_lgbm_calib = calibrator_lgbm.predict(test_result_lgbm)
test_result_xgb_calib = calibrator_xgb.predict(test_result_xgb)

In [ ]:
X_meta_train = np.column_stack([oof_result_lgbm_calib, oof_result_xgb_calib])
X_meta_test = np.column_stack([test_result_lgbm_calib, test_result_xgb_calib])

In [ ]:
meta_model = LogisticRegression(C=1.0, solver='lbfgs', random_state=17)
meta_model.fit(X_meta_train, y_train)
meta_oof_predict = meta_model.predict_proba(X_meta_train)[:, 1]
meta_test_predict = meta_model.predict_proba(X_meta_test)[:, 1]

In [ ]:
probabilities = meta_test_predict.copy()
conditions = [
    probabilities < 0.05,
    (probabilities >= 0.05) & (probabilities <= 0.15),
    (probabilities > 0.15) & (probabilities <= 0.30)
]
choices = ['A_Low_Risk', 'B_Medium_Risk', 'C_High_Risk']
risk_grade = np.select(conditions, choices, default = 'D_Reject')

In [ ]:
MODEL_PATH = f'{PROJECT_PATH}/Models'
os.makedirs(MODEL_PATH, exist_ok=True)

meta_model_path = os.path.join(MODEL_PATH, 'meta_model.joblib')
lgbm_path = os.path.join(MODEL_PATH, 'lgbm_folds.joblib')
xgb_path = os.path.join(MODEL_PATH, 'xgb_folds.joblib')

joblib.dump(meta_model, meta_model_path)
joblib.dump(model_list_lgbm, lgbm_path, compress=3)
joblib.dump(model_xgb_list, xgb_path, compress=3)

In [ ]:
submission = pd.DataFrame({
    'SK_ID_CURR': test_id,
    'TARGET': meta_test_predict,
    'RISK_GRADE': risk_grade
})

output_path = os.path.join(PROJECT_PATH, 'final_credit_decisions.csv')

submission.to_csv(output_path, index=False)